# Currency Exchange Insights — Exploratory Data Analysis

Source: `data/currency.db` (SQLite), built by `data_pipeline/fetch_data.py` → `clean_transform.py` → `load_to_db.py`.

Covers 1 year of trailing daily exchange rates, base currency USD, sourced from the Frankfurter v2 API (ECB + partner central banks).

This notebook covers, per the capstone brief:
1. Time series line plots per currency pair
2. Distribution of daily returns (outliers/extreme moves)
3. Correlation heatmap across currency pairs
4. Rolling volatility comparison
5. Seasonality check (day-of-week / month-end patterns)
6. Top gainers/losers over the year


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DB_PATH = "../data/currency.db"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM exchange_rates", conn, parse_dates=["date"])
conn.close()

print(df.shape)
df.head()

## 0. Data overview

In [ ]:
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Base currency:", df["base_currency"].unique())
print("Target currencies:", sorted(df["target_currency"].unique()))
print("Trading-day rows:", int(df["is_trading_day"].sum()), "/", len(df))
df.isnull().sum()

Rows where `is_trading_day == 0` are weekend/holiday placeholders added by `clean_transform.py`'s
calendar reindex — `rate` and the rolling/derived columns are `NaN` there. Drop them for any analysis
that should only look at actual published rates.

In [ ]:
trading = df[df["is_trading_day"] == 1].copy()
trading = trading.sort_values(["target_currency", "date"])
trading.shape

## 1. Time series line plots per currency pair

In [ ]:
currencies = sorted(trading["target_currency"].unique())
n = len(currencies)
ncols = 2
nrows = -(-n // ncols)  # ceil

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows), sharex=True)
axes = axes.flatten()

for ax, cur in zip(axes, currencies):
    sub = trading[trading["target_currency"] == cur]
    ax.plot(sub["date"], sub["rate"], linewidth=1)
    ax.plot(sub["date"], sub["ma_30"], linewidth=1.2, linestyle="--", label="30d MA")
    ax.set_title(f"USD/{cur}")
    ax.legend(fontsize=8)

for ax in axes[n:]:
    ax.set_visible(False)

fig.suptitle("Exchange rate trend with 30-day moving average, trailing 12 months", y=1.02)
fig.tight_layout()
plt.show()

## 2. Distribution of daily returns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=trading, x="daily_pct_change", hue="target_currency",
             bins=60, element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Distribution of daily % change by currency")
axes[0].set_xlim(-0.05, 0.05)

sns.boxplot(data=trading, x="target_currency", y="daily_pct_change", ax=axes[1])
axes[1].set_title("Daily % change spread by currency (outliers visible as points)")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
plt.show()

In [ ]:
# Flag the most extreme single-day moves across all currencies
extreme_moves = (
    trading.dropna(subset=["daily_pct_change"])
    .reindex(trading["daily_pct_change"].abs().sort_values(ascending=False).index)
    .head(10)[["date", "target_currency", "rate", "daily_pct_change"]]
)
extreme_moves

## 3. Correlation heatmap across currency pairs

In [ ]:
pivot = trading.pivot_table(index="date", columns="target_currency", values="rate")
corr = pivot.pct_change().corr()   # correlate daily returns, not raw levels

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation of daily returns across currency pairs (USD base)")
plt.show()

Correlating daily **returns** (`pct_change`) rather than raw rate levels avoids the spurious
correlation you'd get from two unrelated series that both happen to trend over a year.

## 4. Rolling volatility comparison

In [ ]:
latest_vol = (
    trading.dropna(subset=["volatility_30d"])
    .sort_values("date")
    .groupby("target_currency")
    .tail(1)[["target_currency", "volatility_30d"]]
    .sort_values("volatility_30d", ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=latest_vol, y="target_currency", x="volatility_30d", ax=ax, orient="h")
ax.set_title("Most recent 30-day rolling volatility by currency")
ax.set_xlabel("30-day rolling std. dev. of daily % change")
plt.show()

latest_vol

In [ ]:
# Volatility over time, not just the latest snapshot
fig, ax = plt.subplots(figsize=(12, 5))
for cur in currencies:
    sub = trading[trading["target_currency"] == cur]
    ax.plot(sub["date"], sub["volatility_30d"], label=cur, linewidth=1)
ax.set_title("30-day rolling volatility over the year")
ax.legend(ncol=4, fontsize=8)
plt.show()

## 5. Seasonality check — day-of-week and month-end patterns

In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
dow_effect = (
    trading[trading["day_of_week"].isin(dow_order)]
    .groupby("day_of_week")["daily_pct_change"]
    .mean()
    .reindex(dow_order)
)

fig, ax = plt.subplots(figsize=(7, 4))
dow_effect.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Average daily % change by day of week (all currencies pooled)")
ax.set_ylabel("Mean daily % change")
plt.show()

In [ ]:
month_end_effect = trading.groupby("is_month_end")["daily_pct_change"].agg(["mean", "std", "count"])
month_end_effect.index = month_end_effect.index.map({0: "Regular day", 1: "Month-end"})
month_end_effect

With ~1 year of data and 5 trading days a week, treat any day-of-week or month-end effect here as
a starting hypothesis rather than a statistically robust finding — a proper significance test would need
several years of history to rule out noise.

## 6. Top gainers/losers over the year

In [ ]:
start_end = (
    trading.sort_values("date")
    .groupby("target_currency")
    .agg(start_rate=("rate", "first"), end_rate=("rate", "last"))
)
start_end["pct_change_ytd"] = (start_end["end_rate"] / start_end["start_rate"] - 1) * 100
start_end = start_end.sort_values("pct_change_ytd", ascending=False)
start_end

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#2a9d8f" if v >= 0 else "#e76f51" for v in start_end["pct_change_ytd"]]
ax.barh(start_end.index, start_end["pct_change_ytd"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% change over trailing 12 months (USD base)")
ax.set_title("Best/worst performing currencies vs. USD, trailing 12 months")
plt.show()

## Key findings (fill in after running on the real 1-year pull)

- Best performer vs. USD: **`{best}`** (+{best_pct:.1f}%)
- Worst performer vs. USD: **`{worst}`** ({worst_pct:.1f}%)
- Most volatile currency (latest 30-day window): **`{most_vol}`**
- Least volatile currency: **`{least_vol}`**
- Any notable day-of-week or month-end effect: _describe here once the real data confirms or rules it out_
- Most/least correlated pair: _read off the heatmap above_

*(This cell's numbers are computed live in the next cell — copy them up into this markdown summary for the README's "Key Insights" section.)*

In [ ]:
best = start_end["pct_change_ytd"].idxmax()
worst = start_end["pct_change_ytd"].idxmin()
most_vol = latest_vol.iloc[0]["target_currency"]
least_vol = latest_vol.iloc[-1]["target_currency"]

print(f"Best performer: {best} ({start_end.loc[best, 'pct_change_ytd']:.2f}%)")
print(f"Worst performer: {worst} ({start_end.loc[worst, 'pct_change_ytd']:.2f}%)")
print(f"Most volatile (latest 30d): {most_vol}")
print(f"Least volatile (latest 30d): {least_vol}")